# 15 · Phase 8 — Deep baselines (training order steps 1–4)

All networks share the MST-ResFLNet-style building blocks (pyramid-spectrum stem, SE attention,
residual stages, temporal attention pooling, class-weighted BCE + focal loss) so that the only
difference between rows is the *representation / fusion*, which is what RL will choose.

| step | model | branches | fixed configuration |
|---|---|---|---|
| 1 | raw-waveform CNN | T | 2 s windows |
| 2 | fixed-STFT CNN | S | 20–800 Hz, 2 s |
| 3 | fixed-db6 wavelet CNN | W | db6, level 5, 20–800 Hz, 2 s |
| 4 | fixed temporal–spectral fusion | T+S (attention) | 20–800 Hz, 2 s |
| 4b | fixed tri-branch fusion (no RL) | T+S+W (attention) | the baseline configuration |
| 4c | supernet fine-tuned, fixed configuration | T+S+W (attention) | baseline configuration, initialised from the Notebook-07 supernet |

Row 4c is the fair control for the Phase-13 model of Notebook 16, which starts from the same
supernet: any gain of the RL model over 4c is due to the *policy*, not to the extra pre-training.

Training: random crops (4 per recording per epoch), AdamW + one-cycle LR, bf16 autocast, early
stopping on validation AUROC. Recording-level probability = mean over half-overlapping windows.
The decision threshold is tuned for macro-F1 **on validation** and frozen for the test split.
Three seeds per model.

In [ ]:
%run common/00_config.ipynb
%run common/01_signal_lib.ipynb
%run common/02_rl_lib.ipynb
%run common/03_deep_lib.ipynb
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
space = ConfigSpace(CFG)
meta, signals = load_deep_data(P)
print(meta.split.value_counts().to_dict(), "| positives:", meta.groupby("split").y.sum().to_dict())
base = config_dict(space, space.baseline)


def fixed(**kw):
    return [dict(base, **kw) for _ in range(len(meta))]


BASELINES = {
    "B1_raw_cnn": (fixed(fusion="T"), ("T",)),
    "B2_stft_cnn": (fixed(fusion="S"), ("S",)),
    "B3_db6_wavelet_cnn": (fixed(fusion="W"), ("W",)),
    "B4_fixed_TS_fusion": (fixed(fusion="TS"), ("T", "S")),
    "B5_fixed_TSW_fusion": (fixed(fusion="TSW"), ("T", "S", "W")),
}
FINETUNED = {"B6_supernet_ft_fixed_config": fixed(fusion="TSW")}
rows = []
for name, (configs, branches) in BASELINES.items():
    print(f"=== {name}")
    rows += run_experiment(name, meta, signals, configs, CFG, P, device, seeds=CFG["deep"]["seeds"], branches=branches,
                           log=lambda s: print(s) if s.startswith("[") else None)
for name, configs in FINETUNED.items():
    print(f"=== {name}")
    rows += run_experiment(name, meta, signals, configs, CFG, P, device, seeds=CFG["deep"]["seeds"],
                           init_from="supernet", lr=CFG["deep"]["lr"] / 5, epochs=15, log=lambda s: print(s) if s.startswith("[") else None)
res = pd.DataFrame(rows)
res.to_csv(P["tables"] / "15_deep_baselines_runs.csv", index=False)
mean_std_table([{"model": r["model"], **{k[5:]: v for k, v in r.items() if k.startswith("test_")}} for r in rows])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, len(BASELINES) + len(FINETUNED), figsize=(26, 3.5), sharey=True)
for a, name in zip(ax, list(BASELINES) + list(FINETUNED)):
    for s in CFG["deep"]["seeds"]:
        h = pd.read_csv(P["models"] / name / f"history_s{s}.csv")
        a.plot(h.epoch, h.val_auroc, label=f"seed {s}")
    a.set_title(name, fontsize=9); a.set_xlabel("epoch")
ax[0].set_ylabel("validation AUROC"); ax[0].legend()
plt.tight_layout(); savefig(fig, "15_baseline_training_curves"); plt.show()